# Epileptic Seizure Detection — CHB-MIT Full Pipeline
Memory-safe loading, feature extraction, RF + 1D-CNN training.

In [ ]:
# ============================================================
# STEP 0: SET YOUR DATA PATH
# ============================================================
data_dir = r'E:\Epileptic Seizure Dataset\chb-mit-scalp-eeg-database-1.0.0\chb-mit-scalp-eeg-database-1.0.0'

import os
assert os.path.exists(data_dir), f"Data directory not found: {data_dir}"
print(f"✓ Data directory confirmed: {data_dir}")


In [ ]:
# ============================================================
# STEP 1: IMPORTS
# ============================================================
import mne
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, auc, f1_score, accuracy_score)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import RandomizedSearchCV
from scipy import signal, stats
import joblib
import glob
import re
import gc

print("✓ All libraries imported successfully.")
print(f"  TensorFlow version: {tf.__version__}")
print(f"  MNE version: {mne.__version__}")


In [ ]:
# ============================================================
# STEP 2: CONFIGURATION
# ============================================================
FS          = 256          # Sampling frequency (Hz)
WINDOW_SEC  = 5            # Window duration (seconds)
WINDOW_SIZE = FS * WINDOW_SEC   # Samples per window = 1280
MAX_WINDOWS_PER_FILE = 5000     # Cap normal windows to limit RAM

COMMON_CHANNELS = [
    'FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1',
    'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
    'FZ-CZ',  'CZ-PZ',
    'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
    'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2',
]

CACHE_DIR = 'data_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Config:")
print(f"  Sampling rate : {FS} Hz")
print(f"  Window size   : {WINDOW_SEC}s → {WINDOW_SIZE} samples")
print(f"  Channels      : {len(COMMON_CHANNELS)}")
print(f"  Cache dir     : {CACHE_DIR}/")


In [ ]:
# ============================================================
# STEP 3: PARSE SEIZURE REGISTRY FROM SUMMARY FILES
# ============================================================
def parse_seizure_registry(data_dir):
    seizure_registry = {}
    summary_files = glob.glob(os.path.join(data_dir, '*', '*-summary.txt'))
    print(f"  Found {len(summary_files)} summary files")

    for summary_file in sorted(summary_files):
        with open(summary_file, 'r') as f:
            lines = f.readlines()

        current_file  = None
        pending_start = None

        for line in lines:
            if 'File Name:' in line:
                m = re.search(r'File Name:\s+(\S+\.edf)', line, re.IGNORECASE)
                if m:
                    current_file = m.group(1)
            elif 'Seizure' in line and 'Start Time:' in line:
                m = re.search(r'Start Time:\s+(\d+)\s+seconds', line)
                if m:
                    pending_start = int(m.group(1))
            elif 'Seizure' in line and 'End Time:' in line:
                m = re.search(r'End Time:\s+(\d+)\s+seconds', line)
                if m and pending_start is not None and current_file is not None:
                    seizure_registry.setdefault(current_file, []).append(
                        (pending_start, int(m.group(1)))
                    )
                    pending_start = None

    total = sum(len(v) for v in seizure_registry.values())
    print(f"✓ Parsed {len(seizure_registry)} files with seizures ({total} total seizures)")
    print(f"  Sample: {list(seizure_registry.items())[:3]}")
    return seizure_registry

seizure_registry = parse_seizure_registry(data_dir)


In [ ]:
# ============================================================
# STEP 4: MEMORY-SAFE WINDOW EXTRACTION  (KEY FIX)
# ============================================================
# FIX: raw.load_data() was loading the whole file into RAM.
#      We now use raw[:, start:end] to pull only the needed
#      slice from disk — preload=False stays lazy throughout.

def get_labeled_windows(file_path, is_seizure_file=False, seizure_registry=None):
    try:
        # 1. Open lazily — NO preload
        raw = mne.io.read_raw_edf(file_path, preload=False, verbose=False)

        # 2. Keep only the channels we want
        valid_picks = [ch for ch in COMMON_CHANNELS if ch in raw.ch_names]
        if not valid_picks:
            print(f"  ⚠ No common channels in {os.path.basename(file_path)}, skipping.")
            return _empty(0), np.array([])
        raw.pick(valid_picks)

        # 3. Band-pass filter BEFORE reading any data (MNE applies lazily)
        raw.filter(l_freq=0.5, h_freq=40.0, method='fir', verbose=False)

        n_samples  = raw.n_times
        n_ch       = len(raw.ch_names)
        fname      = os.path.basename(file_path)
        windows, labels = [], []

        if is_seizure_file and seizure_registry and fname in seizure_registry:
            # ── Ictal windows only ──
            for start_s, end_s in seizure_registry[fname]:
                start_sample = int(start_s * FS)
                end_sample   = min(int(end_s * FS), n_samples)
                for i in range(start_sample, end_sample - WINDOW_SIZE, WINDOW_SIZE):
                    data, _ = raw[:, i : i + WINDOW_SIZE]   # ← disk slice, not RAM
                    windows.append(data.flatten().astype(np.float32))
                    labels.append(1)
        else:
            # ── Normal (interictal) windows ──
            count = 0
            for i in range(0, n_samples - WINDOW_SIZE, WINDOW_SIZE):
                if count >= MAX_WINDOWS_PER_FILE:
                    break
                data, _ = raw[:, i : i + WINDOW_SIZE]       # ← disk slice
                windows.append(data.flatten().astype(np.float32))
                labels.append(0)
                count += 1

        del raw
        gc.collect()

        if windows:
            return np.array(windows, dtype=np.float32), np.array(labels, dtype=np.int8)
        return _empty(n_ch), np.array([])

    except MemoryError:
        print(f"  ✗ MemoryError on {os.path.basename(file_path)} — skipping")
        return _empty(0), np.array([])
    except Exception as e:
        print(f"  ✗ Error on {os.path.basename(file_path)}: {e}")
        return _empty(0), np.array([])

def _empty(n_ch):
    cols = WINDOW_SIZE * n_ch if n_ch else WINDOW_SIZE * len(COMMON_CHANNELS)
    return np.zeros((0, cols), dtype=np.float32)

print("✓ get_labeled_windows defined (memory-safe version)")


In [ ]:
# ============================================================
# STEP 5: DISK CACHE HELPERS
# ============================================================
def cache_path_for(fname, quarter):
    base = os.path.splitext(fname)[0]
    return os.path.join(CACHE_DIR, f"{base}_q{quarter}.npz")

def save_to_cache(fname, X, y, quarter):
    path = cache_path_for(fname, quarter)
    np.savez_compressed(path, X=X, y=y)
    return path

def load_from_cache(path):
    d = np.load(path)
    return d['X'], d['y']

print(f"✓ Cache helpers ready. Cache dir: {CACHE_DIR}/")


In [ ]:
# ============================================================
# STEP 6: DISCOVER ALL EDF FILES
# ============================================================
edf_files = sorted(glob.glob(os.path.join(data_dir, '*', '*.edf')))
print(f"✓ Found {len(edf_files)} EDF files")

files_to_load = []
for fp in edf_files:
    fname = os.path.basename(fp)
    files_to_load.append((fp, fname, fname in seizure_registry))

n_seizure = sum(1 for _, _, s in files_to_load if s)
print(f"  Seizure files : {n_seizure}")
print(f"  Normal files  : {len(files_to_load) - n_seizure}")

# Split into 4 quarters for incremental processing
q = len(files_to_load) // 4
quarters = [
    files_to_load[0      : q    ],
    files_to_load[q      : q*2  ],
    files_to_load[q*2    : q*3  ],
    files_to_load[q*3    :      ],
]
for i, qtr in enumerate(quarters, 1):
    print(f"  Quarter {i}: {len(qtr)} files")


In [ ]:
# ============================================================
# QUARTER 1: PROCESS & CACHE TO DISK
# ============================================================
cached_paths_q1 = []

for file_path, fname, is_seizure in quarters[0]:
    cp = cache_path_for(fname, 1)
    if os.path.exists(cp):
        print(f"  ↩ {fname}: already cached, skipping")
        cached_paths_q1.append(cp)
        continue

    X, y = get_labeled_windows(file_path,
                               is_seizure_file=is_seizure,
                               seizure_registry=seizure_registry)
    if len(X) > 0:
        cp = save_to_cache(fname, X, y, 1)
        cached_paths_q1.append(cp)
        print(f"  ✓ {fname}: {len(y)} windows ({int(y.sum())} seizure) → cached")
    else:
        print(f"  ✗ {fname}: no windows extracted")

    del X, y
    gc.collect()

print(f"\n✓ Quarter 1: {len(cached_paths_q1)} files cached")


In [ ]:
# ============================================================
# QUARTER 2: PROCESS & CACHE TO DISK
# ============================================================
cached_paths_q2 = []

for file_path, fname, is_seizure in quarters[1]:
    cp = cache_path_for(fname, 2)
    if os.path.exists(cp):
        print(f"  ↩ {fname}: already cached, skipping")
        cached_paths_q2.append(cp)
        continue

    X, y = get_labeled_windows(file_path,
                               is_seizure_file=is_seizure,
                               seizure_registry=seizure_registry)
    if len(X) > 0:
        cp = save_to_cache(fname, X, y, 2)
        cached_paths_q2.append(cp)
        print(f"  ✓ {fname}: {len(y)} windows ({int(y.sum())} seizure) → cached")
    else:
        print(f"  ✗ {fname}: no windows extracted")

    del X, y
    gc.collect()

print(f"\n✓ Quarter 2: {len(cached_paths_q2)} files cached")


In [ ]:
# ============================================================
# QUARTER 3: PROCESS & CACHE TO DISK
# ============================================================
cached_paths_q3 = []

for file_path, fname, is_seizure in quarters[2]:
    cp = cache_path_for(fname, 3)
    if os.path.exists(cp):
        print(f"  ↩ {fname}: already cached, skipping")
        cached_paths_q3.append(cp)
        continue

    X, y = get_labeled_windows(file_path,
                               is_seizure_file=is_seizure,
                               seizure_registry=seizure_registry)
    if len(X) > 0:
        cp = save_to_cache(fname, X, y, 3)
        cached_paths_q3.append(cp)
        print(f"  ✓ {fname}: {len(y)} windows ({int(y.sum())} seizure) → cached")
    else:
        print(f"  ✗ {fname}: no windows extracted")

    del X, y
    gc.collect()

print(f"\n✓ Quarter 3: {len(cached_paths_q3)} files cached")


In [ ]:
# ============================================================
# QUARTER 4: PROCESS & CACHE TO DISK
# ============================================================
cached_paths_q4 = []

for file_path, fname, is_seizure in quarters[3]:
    cp = cache_path_for(fname, 4)
    if os.path.exists(cp):
        print(f"  ↩ {fname}: already cached, skipping")
        cached_paths_q4.append(cp)
        continue

    X, y = get_labeled_windows(file_path,
                               is_seizure_file=is_seizure,
                               seizure_registry=seizure_registry)
    if len(X) > 0:
        cp = save_to_cache(fname, X, y, 4)
        cached_paths_q4.append(cp)
        print(f"  ✓ {fname}: {len(y)} windows ({int(y.sum())} seizure) → cached")
    else:
        print(f"  ✗ {fname}: no windows extracted")

    del X, y
    gc.collect()

print(f"\n✓ Quarter 4: {len(cached_paths_q4)} files cached")


In [ ]:
# ============================================================
# STEP 7: ASSEMBLE DATASET VIA MEMORY-MAPPED ARRAYS  (KEY FIX)
# ============================================================
# FIX: Previously all cache chunks were loaded into a Python
#      list and then np.concatenated — loading everything into
#      RAM at once.  We now pre-allocate disk-backed memmaps
#      and fill them chunk by chunk.

all_cached_paths = (cached_paths_q1 + cached_paths_q2 +
                    cached_paths_q3 + cached_paths_q4)
print(f"Assembling {len(all_cached_paths)} cached chunks...")

# ── Pass 1: count total windows & infer feature width ──
total_windows = 0
feature_width = None
for cp in all_cached_paths:
    d = np.load(cp)
    total_windows += len(d['y'])
    if feature_width is None:
        feature_width = d['X'].shape[1]

print(f"  Total windows : {total_windows:,}")
print(f"  Feature width : {feature_width}  ({feature_width // WINDOW_SIZE} channels × {WINDOW_SIZE} samples)")
print(f"  Estimated size: ~{total_windows * feature_width * 4 / 1024**3:.2f} GB")

# ── Pass 2: fill memory-mapped files ──
X      = np.lib.format.open_memmap('X_all.npy', mode='w+', dtype=np.float32,
                                    shape=(total_windows, feature_width))
y_arr  = np.lib.format.open_memmap('y_all.npy', mode='w+', dtype=np.float32,
                                    shape=(total_windows,))
groups = np.zeros(total_windows, dtype=np.int32)

offset = 0
for file_idx, cp in enumerate(all_cached_paths):
    d = np.load(cp)
    Xc, yc = d['X'], d['y']
    n = len(yc)
    X[offset:offset+n]      = Xc
    y_arr[offset:offset+n]  = yc
    groups[offset:offset+n] = file_idx
    offset += n
    print(f"  [{file_idx+1}/{len(all_cached_paths)}] {os.path.basename(cp)}: {n} windows")
    del Xc, yc
    gc.collect()

y = y_arr  # alias

print(f"\n✓ Dataset assembled:")
print(f"  X shape      : {X.shape}")
print(f"  Normal        : {int((y==0).sum()):,}")
print(f"  Seizure       : {int((y==1).sum()):,}")
print(f"  Unique files  : {len(np.unique(groups))}")


In [ ]:
# ============================================================
# STEP 8: PATIENT-AWARE TRAIN/TEST SPLIT
# ============================================================
# GroupShuffleSplit ensures no patient's data appears in both
# train and test sets — critical for unbiased evaluation.

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_raw = X[train_idx]
X_test_raw  = X[test_idx]
y_train     = y[train_idx].astype(int)
y_test      = y[test_idx].astype(int)

# Scale on training data ONLY (prevent data leakage)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled  = scaler.transform(X_test_raw)

joblib.dump(scaler, 'scaler.pkl')

print(f"Train/Test split (file-aware GroupShuffleSplit):")
print(f"  Train : {len(y_train):,}  (Normal={int((y_train==0).sum()):,}, Seizure={int((y_train==1).sum()):,})")
print(f"  Test  : {len(y_test):,}  (Normal={int((y_test==0).sum()):,}, Seizure={int((y_test==1).sum()):,})")
print(f"\n✓ scaler.pkl saved")


In [ ]:
# ============================================================
# STEP 9: FEATURE EXTRACTION FOR RANDOM FOREST
# ============================================================
# FIX: extract_features was defined only in app.py, not here.
#      Also upgraded with band-power features for better RF accuracy.

N_CHANNELS = X_train_scaled.shape[1] // WINDOW_SIZE

def extract_features_batch(X_flat, window_size=WINDOW_SIZE, n_ch=N_CHANNELS, sfreq=FS):
    """
    Extract statistical + spectral band-power features per window.
    Input : (n_windows, window_size * n_channels) flattened
    Output: (n_windows, n_features)
    """
    n_windows = X_flat.shape[0]
    X_3d = X_flat.reshape(n_windows, window_size, n_ch)
    features = []

    for i in range(n_windows):
        win_feats = []
        for ch in range(n_ch):
            ch_data = X_3d[i, :, ch].astype(np.float64)

            # ── Time-domain features ──
            win_feats += [
                ch_data.mean(),
                ch_data.std(),
                ch_data.var(),
                np.percentile(ch_data, 25),
                np.percentile(ch_data, 75),
                float(stats.skew(ch_data)),
                float(stats.kurtosis(ch_data)),
                ch_data.min(),
                ch_data.max(),
            ]

            # ── Spectral band-power features ──
            freqs, psd = signal.welch(ch_data, sfreq, nperseg=min(sfreq, window_size))
            win_feats += [
                float(psd[(freqs >= 0.5) & (freqs < 4 )].mean()),   # Delta
                float(psd[(freqs >= 4  ) & (freqs < 8 )].mean()),   # Theta
                float(psd[(freqs >= 8  ) & (freqs < 13)].mean()),   # Alpha
                float(psd[(freqs >= 13 ) & (freqs < 30)].mean()),   # Beta
                float(psd[(freqs >= 30 ) & (freqs < 50)].mean()),   # Low Gamma
            ]
        features.append(win_feats)

    return np.array(features, dtype=np.float32)

print(f"Extracting features for {len(X_train_scaled):,} training windows...")
print(f"  (This takes a few minutes — {N_CHANNELS} channels × 14 features each)")
X_train_feat = extract_features_batch(X_train_scaled)

print(f"Extracting features for {len(X_test_scaled):,} test windows...")
X_test_feat = extract_features_batch(X_test_scaled)

print(f"\n✓ Feature matrices ready:")
print(f"  X_train_feat : {X_train_feat.shape}")
print(f"  X_test_feat  : {X_test_feat.shape}")


In [ ]:
# ============================================================
# STEP 10: CLASS WEIGHTS  (handles severe imbalance)
# ============================================================
class_weights_arr = compute_class_weight('balanced',
                                          classes=np.unique(y_train),
                                          y=y_train)
class_weights_dict = {int(cls): float(w)
                      for cls, w in zip(np.unique(y_train), class_weights_arr)}

print(f"Class weights: {class_weights_dict}")
print(f"  → The model will treat each seizure window as {class_weights_dict.get(1,1):.1f}× "
      f"more important than a normal window.")


In [ ]:
# ============================================================
# STEP 11: TRAIN RANDOM FOREST BASELINE
# ============================================================
rf_baseline = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
rf_baseline.fit(X_train_feat, y_train)

rf_test_pred = rf_baseline.predict(X_test_feat)
rf_test_prob = rf_baseline.predict_proba(X_test_feat)[:, 1]
rf_train_acc = accuracy_score(y_train, rf_baseline.predict(X_train_feat))
rf_test_acc  = accuracy_score(y_test, rf_test_pred)
rf_f1        = f1_score(y_test, rf_test_pred, zero_division=0)
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_test_prob)
rf_auc       = auc(rf_fpr, rf_tpr)

print(f"\n✓ Random Forest Baseline:")
print(f"  Train accuracy : {rf_train_acc:.4f}")
print(f"  Test accuracy  : {rf_test_acc:.4f}  (gap: {rf_train_acc-rf_test_acc:.4f})")
print(f"  F1-score       : {rf_f1:.4f}")
print(f"  ROC-AUC        : {rf_auc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, rf_test_pred,
      target_names=['Normal', 'Seizure'], zero_division=0))
print(f"Confusion Matrix:\n{confusion_matrix(y_test, rf_test_pred)}")


In [ ]:
# ============================================================
# STEP 12: RANDOM FOREST HYPERPARAMETER TUNING
# ============================================================
param_dist = {
    'n_estimators'    : [100, 200, 300],
    'max_depth'       : [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
    'bootstrap'       : [True, False],
}

rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1,
)
rf_search.fit(X_train_feat, y_train)

best_rf = rf_search.best_estimator_
best_rf_pred = best_rf.predict(X_test_feat)
best_rf_prob = best_rf.predict_proba(X_test_feat)[:, 1]
best_rf_acc  = accuracy_score(y_test, best_rf_pred)
best_rf_f1   = f1_score(y_test, best_rf_pred, zero_division=0)
best_rf_fpr, best_rf_tpr, _ = roc_curve(y_test, best_rf_prob)
best_rf_auc  = auc(best_rf_fpr, best_rf_tpr)

print(f"\n✓ Best parameters: {rf_search.best_params_}")
print(f"  Test accuracy  : {best_rf_acc:.4f}")
print(f"  F1-score       : {best_rf_f1:.4f}")
print(f"  ROC-AUC        : {best_rf_auc:.4f}")
print(classification_report(y_test, best_rf_pred,
      target_names=['Normal', 'Seizure'], zero_division=0))

joblib.dump(best_rf, 'best_rf_tuned_model.pkl')
print("\n✅ best_rf_tuned_model.pkl saved")


In [ ]:
# ============================================================
# STEP 13: RESHAPE DATA FOR 1D-CNN
# ============================================================
# FIX: use dynamic n_channels instead of hardcoded len(COMMON_CHANNELS)
#      to avoid shape mismatch when some files have fewer channels.

N_CH_CNN = X_train_scaled.shape[1] // WINDOW_SIZE   # actual channels in data

X_train_cnn = X_train_scaled.reshape(-1, WINDOW_SIZE, N_CH_CNN)
X_test_cnn  = X_test_scaled.reshape(-1, WINDOW_SIZE, N_CH_CNN)

print(f"CNN input shapes:")
print(f"  X_train_cnn : {X_train_cnn.shape}  (windows × timesteps × channels)")
print(f"  X_test_cnn  : {X_test_cnn.shape}")


In [ ]:
# ============================================================
# STEP 14: BUILD 1D-CNN
# ============================================================
def build_cnn(input_shape):
    model = models.Sequential([
        layers.Conv1D(16, kernel_size=3, activation='relu',
                      input_shape=input_shape,
                      kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),

        layers.Conv1D(32, kernel_size=3, activation='relu',
                      kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),

        layers.Conv1D(64, kernel_size=3, activation='relu',
                      kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling1D(),

        layers.Dense(64, activation='relu',
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.AUC(name='auc'),
                 tf.keras.metrics.Recall(name='sensitivity')],
    )
    return model

advanced_model = build_cnn((WINDOW_SIZE, N_CH_CNN))
advanced_model.summary()


In [ ]:
# ============================================================
# STEP 15: TRAIN 1D-CNN
# ============================================================
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        'best_cnn.keras', monitor='val_auc', save_best_only=True,
        mode='max', verbose=0),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1),
]

history = advanced_model.fit(
    X_train_cnn, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights_dict,
    callbacks=callbacks,
    verbose=1,
)


In [ ]:
# ============================================================
# STEP 16: EVALUATE 1D-CNN
# ============================================================
cnn_test_prob = advanced_model.predict(X_test_cnn, verbose=0).flatten()
cnn_test_pred = (cnn_test_prob > 0.5).astype(int)
cnn_train_pred = (advanced_model.predict(X_train_cnn, verbose=0).flatten() > 0.5).astype(int)

cnn_train_acc = accuracy_score(y_train, cnn_train_pred)
cnn_test_acc  = accuracy_score(y_test,  cnn_test_pred)
cnn_f1        = f1_score(y_test, cnn_test_pred, zero_division=0)
cnn_fpr, cnn_tpr, _ = roc_curve(y_test, cnn_test_prob)
cnn_auc       = auc(cnn_fpr, cnn_tpr)

print(f"\n✓ 1D-CNN Results:")
print(f"  Train accuracy : {cnn_train_acc:.4f}")
print(f"  Test accuracy  : {cnn_test_acc:.4f}  (gap: {cnn_train_acc-cnn_test_acc:.4f})")
print(f"  F1-score       : {cnn_f1:.4f}")
print(f"  ROC-AUC        : {cnn_auc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, cnn_test_pred,
      target_names=['Normal', 'Seizure'], zero_division=0))
print(f"Confusion Matrix:\n{confusion_matrix(y_test, cnn_test_pred)}")


In [ ]:
# ============================================================
# STEP 17: MODEL COMPARISON & EXPORT
# ============================================================
print(f"{'Metric':<22} {'RF (tuned)':<18} {'1D-CNN':<18}")
print("-" * 58)
print(f"{'Train Accuracy':<22} {best_rf_acc:<18.4f} {cnn_train_acc:<18.4f}")
print(f"{'Test Accuracy':<22} {best_rf_acc:<18.4f} {cnn_test_acc:<18.4f}")
print(f"{'Overfitting Gap':<22} {accuracy_score(y_train, best_rf.predict(X_train_feat)) - best_rf_acc:<18.4f} {cnn_train_acc - cnn_test_acc:<18.4f}")
print(f"{'F1-Score':<22} {best_rf_f1:<18.4f} {cnn_f1:<18.4f}")
print(f"{'ROC-AUC':<22} {best_rf_auc:<18.4f} {cnn_auc:<18.4f}")

better_model = '1D-CNN' if cnn_test_acc > best_rf_acc else 'Random Forest'
print(f"\n✓ Best model: {better_model}")

# ── Save everything ──
advanced_model.save('advanced_model.h5')
joblib.dump(history.history, 'training_history.pkl')

metrics = {
    'dataset_info': {
        'total_windows'  : int(len(y)),
        'normal_samples' : int((y == 0).sum()),
        'seizure_samples': int((y == 1).sum()),
        'num_files'      : int(len(np.unique(groups))),
        'data_source'    : 'CHB-MIT full dataset',
    },
    'train_test_split': {
        'train_samples': int(len(y_train)),
        'test_samples' : int(len(y_test)),
        'split_method' : 'GroupShuffleSplit (patient-aware)',
    },
    'random_forest': {
        'test_acc' : float(best_rf_acc),
        'f1_score' : float(best_rf_f1),
        'roc_auc'  : float(best_rf_auc),
    },
    'cnn_1d': {
        'test_acc' : float(cnn_test_acc),
        'f1_score' : float(cnn_f1),
        'roc_auc'  : float(cnn_auc),
    },
    'best_model': better_model,
}
joblib.dump(metrics, 'model_metrics.pkl')

print("\n✅ All files saved:")
print("   scaler.pkl               — RobustScaler")
print("   best_rf_tuned_model.pkl  — Tuned Random Forest")
print("   advanced_model.h5        — 1D-CNN")
print("   training_history.pkl     — CNN training curves")
print("   model_metrics.pkl        — Performance metrics")
